In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!apt-get -qq install -y default-jre-headless > /dev/null
!pip -q install openai-whisper language-tool-python spacy sentence-transformers lightgbm
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.2 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.2 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os, glob, torch
import pandas as pd, numpy as np

print('GPU available:', torch.cuda.is_available())

BASE = None
for d, _, f in os.walk('/kaggle/input'):
    if 'train.csv' in f:
        BASE = d; break
assert BASE, 'train.csv not found. Attach the competition under Input.'
print('BASE =', BASE)

train = pd.read_csv(f'{BASE}/train.csv'); test = pd.read_csv(f'{BASE}/test.csv')
sample_sub = pd.read_csv(f'{BASE}/sample_submission.csv')
print(train.shape, test.shape, sample_sub.shape)
print('sample submission columns:', sample_sub.columns.tolist())

wav_dirs = {}
for d, _, f in os.walk('/kaggle/input'):
    n = sum(x.lower().endswith('.wav') for x in f)
    if n: wav_dirs[d] = n

def best_dir(names):
    best, bn = None, -1
    for d in wav_dirs:
        n = names.isin(set(os.listdir(d))).sum()
        if n > bn: best, bn = d, n
    return best, bn

TRAIN_AUDIO, a = best_dir(train['filename'])
TEST_AUDIO,  b = best_dir(test['filename'])
print('TRAIN_AUDIO =', TRAIN_AUDIO, '| found', a, 'of', len(train))
print('TEST_AUDIO  =', TEST_AUDIO,  '| found', b, 'of', len(test))
assert a == len(train) and b == len(test), 'Some audio files are missing. Paste this output to get help.'
assert TRAIN_AUDIO != TEST_AUDIO, 'Train and test resolved to the same folder. Paste this output to get help.'

CACHE = '/kaggle/working/transcripts.csv'
RESET_CACHE = False   # False = reuse existing transcripts (saves time). True = start transcription from scratch.
if RESET_CACHE and os.path.exists(CACHE):
    os.remove(CACHE); print('cache deleted')
print('cache exists:', os.path.exists(CACHE))

GPU available: True
BASE = /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final
(769, 2) (216, 2) (204, 2)
sample submission columns: ['filename', 'label']
TRAIN_AUDIO = /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train | found 769 of 769
TEST_AUDIO  = /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test | found 216 of 216
cache exists: False


In [3]:
import os, torch
import numpy as np, pandas as pd
import whisper
from tqdm import tqdm

for v in ['train', 'test', 'TRAIN_AUDIO', 'TEST_AUDIO', 'CACHE']:
    assert v in globals(), f'{v} is missing. Run Cell 2 first.'
assert torch.cuda.is_available(), 'GPU is OFF. Turn on the accelerator in Session options, then rerun Cells 1 and 2.'

asr = whisper.load_model('small')
PROMPT = "Umm, let me think, like, hmm... Okay, so, I I went to the, uh, the store yesterday and and I think it was good."

def transcribe(path):
    audio = whisper.load_audio(path); dur = len(audio) / 16000
    r = asr.transcribe(audio, language='en', fp16=True, word_timestamps=True,
                       initial_prompt=PROMPT, condition_on_previous_text=False, temperature=0)
    words = [w for s in r['segments'] for w in s.get('words', [])]
    st = np.array([w['start'] for w in words]); en = np.array([w['end'] for w in words])
    gaps = (st[1:] - en[:-1]) if len(words) > 1 else np.array([0.0])
    long_gaps = gaps[gaps > 0.5]
    segs = r['segments']
    return dict(text=r['text'].strip(), duration=dur, n_asr_words=len(words),
                n_pauses=len(long_gaps), pause_total=float(long_gaps.sum()),
                pause_mean=float(long_gaps.mean()) if len(long_gaps) else 0.0,
                speech_time=float(en[-1] - st[0]) if len(words) else 0.0,
                asr_logprob=float(np.mean([s['avg_logprob'] for s in segs])) if segs else -3.0,
                no_speech=float(np.mean([s['no_speech_prob'] for s in segs])) if segs else 1.0,
                compression=float(np.mean([s['compression_ratio'] for s in segs])) if segs else 0.0)

def placeholder(path):
    # used only if a file cannot be read, so the pipeline never crashes
    return dict(text='', duration=50.0, n_asr_words=0, n_pauses=0, pause_total=0.0, pause_mean=0.0,
                speech_time=0.0, asr_logprob=-3.0, no_speech=1.0, compression=0.0)

# Load existing cache. Older caches have no 'split' column: names in train.csv came from the train folder.
if os.path.exists(CACHE):
    old = pd.read_csv(CACHE)
    if 'split' not in old.columns:
        old['split'] = np.where(old.filename.isin(train.filename), 'train', 'test')
else:
    old = pd.DataFrame(columns=['split', 'filename'])
rows = old.to_dict('records')
seen = set(zip(old['split'], old['filename']))
print('already done:', len(seen))

items = [('train', f, f'{TRAIN_AUDIO}/{f}') for f in train.filename] + \
        [('test',  f, f'{TEST_AUDIO}/{f}')  for f in test.filename]
failed = []
for i, (sp, f, p) in enumerate(tqdm(items)):
    if (sp, f) in seen: continue
    try:
        rows.append(dict(split=sp, filename=f, **transcribe(p)))
    except Exception as e:
        failed.append((sp, f)); print('FAILED', sp, f, '->', str(e)[-120:])
        rows.append(dict(split=sp, filename=f, **placeholder(p)))
    if i % 25 == 0: pd.DataFrame(rows).to_csv(CACHE, index=False)

tr = pd.DataFrame(rows).drop_duplicates(['split', 'filename'], keep='last').reset_index(drop=True)
tr.to_csv(CACHE, index=False)
tr['text'] = tr['text'].fillna('')
# put rows in a fixed order: train rows in train.csv order, then test rows in test.csv order
order = pd.concat([pd.DataFrame({'split': 'train', 'filename': train.filename}),
                   pd.DataFrame({'split': 'test',  'filename': test.filename})], ignore_index=True)
tr = order.merge(tr, on=['split', 'filename'], how='left')
assert tr['duration'].notna().all(), 'Some files are missing from the cache. Rerun this cell.'
print(tr.split.value_counts().to_dict(), '| failed files:', len(failed))
if failed: print('WARNING: these files used placeholder rows:', failed)
tr.head()

100%|████████████████████████████████████████| 461M/461M [00:01<00:00, 295MiB/s]


already done: 0


100%|██████████| 985/985 [44:05<00:00,  2.69s/it]

{'train': 769, 'test': 216} | failed files: 0


,split,filename,text,duration,n_asr_words,n_pauses,pause_total,pause_mean,speech_time,asr_logprob,no_speech,compression
0,train,audio_192.wav,"Well, the playground is very, very beautiful a...",60.508375,94,8,11.36,1.420000,53.64,-0.706581,0.142616,1.425028
1,train,audio_436.wav,"The scene of a school playground, our playgrou...",60.074688,110,3,2.40,0.800000,57.14,-0.346206,0.301637,1.467759
2,train,audio_447.wav,My favorite place to visit is the beach. I lov...,60.074688,144,4,2.66,0.665000,54.56,-0.242372,0.351531,1.626053
3,train,audio_126.wav,"Once upon a time, we, with five friends, went ...",45.060000,106,2,2.48,1.240000,44.00,-0.376008,0.223159,1.734975
4,train,audio_61.wav,"Well, when I was a child, I remember I went to...",45.300000,79,3,3.02,1.006667,42.34,-0.289905,0.171229,1.608317


In [10]:
import re
import numpy as np, pandas as pd
import language_tool_python
from tqdm import tqdm

for v in ['tr', 'train', 'test', 'sample_sub']:
    assert v in globals(), f'{v} is missing. Run Cells 1, 2 and 3 first.'

tool = language_tool_python.LanguageTool('en-US')

def text_features(text):
    low = text.lower()
    words = re.findall(r"[a-z']+", low)
    nw = max(len(words), 1)
    sents = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    errs = tool.check(text) if text.strip() else []
    grammar_errs = sum(m.category == 'GRAMMAR' for m in errs)
    return dict(
        grammar_errors_per_100w = 100 * grammar_errs / nw,
        avg_sentence_len        = nw / max(len(sents), 1),
        vocab_diversity         = len(set(words)) / nw,
        fillers_per_100w        = 100 * len(re.findall(r'\b(um+|uh+|hmm+|er|erm|ah)\b', low)) / nw,
        repeats_per_100w        = 100 * len(re.findall(r'\b(\w+)\s+\1\b', low)) / nw,
        n_words                 = nw)

X = pd.DataFrame([text_features(t) for t in tqdm(tr.text)])
X['words_per_min'] = tr.n_asr_words.values / tr.duration.values * 60
X['pause_fraction'] = tr.pause_total.values / tr.duration.values
X['asr_confidence'] = tr.asr_logprob.values
X = X.replace([np.inf, -np.inf], 0).fillna(0).astype(float)
print(X.shape); X.head()

100%|██████████| 985/985 [01:02<00:00, 15.79it/s]

(985, 9)


,grammar_errors_per_100w,avg_sentence_len,vocab_diversity,fillers_per_100w,repeats_per_100w,n_words,words_per_min,pause_fraction,asr_confidence
0,0.000000,9.400000,0.670213,0.0,0.000000,94.0,93.210237,0.187743,-0.706581
1,0.909091,22.000000,0.463636,0.0,0.000000,110.0,109.863243,0.039950,-0.346206
2,1.388889,12.000000,0.520833,0.0,0.000000,144.0,143.820973,0.044278,-0.242372
3,1.886792,35.333333,0.433962,0.0,0.943396,106.0,141.145140,0.055038,-0.376008
4,1.265823,19.750000,0.569620,0.0,2.531646,79.0,104.635762,0.066667,-0.289905


In [11]:
import numpy as np, pandas as pd
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

for v in ['X', 'tr', 'train', 'test']:
    assert v in globals(), f'{v} is missing. Run the earlier cells first.'

is_train = (tr.split == 'train').values
assert is_train.sum() == len(train) and (~is_train).sum() == len(test)
y = train.set_index('filename').label.loc[tr.filename[is_train]].values
Xtr, Xte = X[is_train], X[~is_train]

kf = KFold(5, shuffle=True, random_state=42)
models = {
    'Ridge (linear)': make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20))),
    'Random Forest':  RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1),
}

print(f'Baseline (always predict the average): RMSE {np.std(y):.3f}\n')
scores = {}
for name, m in models.items():
    p = np.clip(cross_val_predict(m, Xtr, y, cv=kf), 0, 5)
    rmse = np.sqrt(np.mean((p - y) ** 2))
    scores[name] = rmse
    print(f'{name:16s} RMSE {rmse:.3f} | MAE {np.mean(np.abs(p - y)):.3f} | Pearson {np.corrcoef(p, y)[0, 1]:.3f}')

best = min(scores, key=scores.get)
print('\nChosen model:', best)

Baseline (always predict the average): RMSE 1.238

Ridge (linear)   RMSE 0.838 | MAE 0.645 | Pearson 0.736
Random Forest    RMSE 0.883 | MAE 0.689 | Pearson 0.701

Chosen model: Ridge (linear)


In [12]:
import numpy as np, pandas as pd
for v in ['models', 'best', 'Xtr', 'Xte', 'y', 'tr', 'is_train', 'test', 'train', 'sample_sub']:
    assert v in globals(), f'{v} is missing. Run Cells 1, 2, 3, A and B first.'

model = models[best].fit(Xtr, y)

if best.startswith('Ridge'):
    print('Ridge coefficients (standardised):')
    print(pd.Series(model[-1].coef_, Xtr.columns).sort_values().round(3))
else:
    print('Random Forest feature importance:')
    print(pd.Series(model.feature_importances_, Xtr.columns).sort_values(ascending=False).round(3))

final = np.clip(model.predict(Xte), 0, 5)
te_names = tr.filename[~is_train].values
assert len(set(te_names)) == len(test)
pred_by_name = pd.Series(final, index=te_names)

id_col, target_col = sample_sub.columns[0], sample_sub.columns[1]
sub = pd.DataFrame({id_col: test.filename.values})   # IDs from test.csv (the sample file's IDs do not match it)
sub[target_col] = pred_by_name.loc[sub[id_col]].values
assert sub[target_col].notna().all() and len(sub) == len(test)
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub.shape); print(sub[target_col].describe().round(2)); sub.head()

Ridge coefficients (standardised):
words_per_min             -0.329
grammar_errors_per_100w   -0.081
avg_sentence_len          -0.054
repeats_per_100w           0.004
fillers_per_100w           0.029
pause_fraction             0.053
vocab_diversity            0.595
n_words                    0.695
asr_confidence             0.735
dtype: float64
(216, 2)
count    216.00
mean       3.40
std        0.71
min        0.33
25%        2.94
50%        3.42
75%        3.83
max        5.00
Name: label, dtype: float64


,filename,label
0,audio_128.wav,2.914561
1,audio_156.wav,4.395060
2,audio_19.wav,4.203868
3,audio_108.wav,2.846543
4,audio_206.wav,3.051468


In [14]:
import pandas as pd

# 1. Submission file
chk = pd.read_csv('/kaggle/working/submission.csv')
print(chk.shape, chk.columns.tolist(), '| duplicates:', chk.filename.duplicated().sum())
print('missing vs test.csv:', len(set(test.filename) - set(chk.filename)))
print('label range:', round(float(chk.label.min()), 2), 'to', round(float(chk.label.max()), 2),
      '| std:', round(float(chk.label.std()), 2), '| train label std:', round(float(y.std()), 2))

# 2. Train and test share file names. Different recordings must never have identical transcripts.
a = tr[tr.split == 'train'].set_index('filename').text
b = tr[tr.split == 'test'].set_index('filename').text
common = a.index.intersection(b.index)
same = (a.loc[common] == b.loc[common]) & (a.loc[common].str.len() > 0)
print('\nshared names:', len(common), '| identical transcripts:', int(same.sum()), '(want 0)')
print('empty transcripts:', int((tr.text.str.len() == 0).sum()), '(want 0 or very few)')

# 3. One-feature-at-a-time correlation with the score (safer to quote than model coefficients)
print('\nCorrelation with the grammar score:')
print(Xtr.corrwith(pd.Series(y, index=Xtr.index)).sort_values().round(2))


(216, 2) ['filename', 'label'] | duplicates: 0
missing vs test.csv: 0
label range: 0.33 to 5.0 | std: 0.71 | train label std: 1.24

shared names: 212 | identical transcripts: 0 (want 0)
empty transcripts: 0 (want 0 or very few)

Correlation with the grammar score:
grammar_errors_per_100w   -0.12
avg_sentence_len          -0.11
pause_fraction            -0.08
repeats_per_100w          -0.08
fillers_per_100w          -0.05
vocab_diversity            0.22
words_per_min              0.24
n_words                    0.28
asr_confidence             0.57
dtype: float64
